# 04c - Java Intricacies (Java)

This is the thinnest notebook in the curriculum, on purpose — `concept.md` is the real content for this topic. This notebook is just a small, hands-on look at the garbage collector actually doing something, plus the discussion questions from `concept.md` for reference.

## Watching the Garbage Collector Do Something

We'll allocate a large number of short-lived objects that nothing keeps a reference to after the loop ends, check how much heap memory the JVM reports as free, then *suggest* a garbage collection pass and check again. `System.gc()` is only a hint to the JVM — it's allowed to ignore it, delay it, or collect more or less than you'd expect — so treat the exact numbers below as illustrative, not guaranteed. Run this cell a few times and notice the numbers move around, which is itself the point: you don't control this the way you control `delete` in C++.

In [1]:
Runtime runtime = Runtime.getRuntime();

long freeBefore = runtime.freeMemory();

// Allocate a lot of short-lived objects. Nothing outside this loop keeps a
// reference to any of them once the loop ends -- they're all GC-eligible
// the moment `garbage` goes out of scope, with no delete anywhere.
for (int i = 0; i < 2_000_000; i++) {
    String garbage = "fault-code-" + i;
}

long freeAfterAllocating = runtime.freeMemory();

System.gc();  // a REQUEST, not a command -- the JVM decides what to actually do with it

long freeAfterGcHint = runtime.freeMemory();

System.out.println("free memory before allocating:     " + freeBefore);
System.out.println("free memory after allocating:      " + freeAfterAllocating);
System.out.println("free memory after System.gc() hint: " + freeAfterGcHint);

free memory before allocating:     58761160


free memory after allocating:      66045656


free memory after System.gc() hint: 73700808


Notice there is no code anywhere above that frees the two million `String` objects we just created — no `delete`, no destructor, nothing. Compare this to `04a_cpp_intricacies`'s `memory_demo.cpp`, where forgetting even one `delete` created a real, permanent leak for the rest of the program's run. Here, forgetting to free anything is the *normal*, correct way to write the code — the JVM's garbage collector is the thing actually responsible for reclaiming that memory, on its own schedule, not yours.

## Discussion Questions

Same three questions from `concept.md` — work through these with a mentor or the rest of your subteam rather than trying to "solve" them here:

1. If Java never lets you choose when memory is freed, how would you even begin debugging a Java program that's slowly using more and more memory over time (a "memory leak," even without a `delete` keyword to forget)? What would still have to be true about your code for that to happen?
2. `04a_cpp_intricacies` argued C++'s manual memory management matters for edge inference because of predictable timing and a tight memory budget. Robot orchestration code (Java, on the roboRIO) also runs on constrained hardware with a real time budget. Why does the tradeoff come out differently for that code than it does for the inference pipeline?
3. Every `Motor` object in `02_oop_inheritance`'s Java notebook is heap-allocated by the JVM, yet nothing in that notebook ever looks like C++'s pointers or references from that same topic. What is the JVM doing on your behalf, every time you write `Motor motor = new TalonMotor(...)`, that C++ makes you do by hand instead?

## Resources

- [Oracle: Garbage Collection Basics](https://docs.oracle.com/javase/8/docs/technotes/guides/vm/gctuning/basics.html) - a real (more detailed than you need yet) look at how the JVM's garbage collector actually works.
- [Runtime (Java SE documentation)](https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/Runtime.html) - `freeMemory()`, `totalMemory()`, and friends, used above.